# Volet A · Fine-tuning de Legal-BERT sur LEDGAR (jalon m4)

**Ce que fait ce notebook**
1. Fine-tuning **complet** de Legal-BERT (Trainer Hugging Face)
2. Fine-tuning **LoRA** avec r = 4, 8, 16
3. **Tableau de compromis** : paramètres entraînables, temps, mémoire GPU, accuracy, macro-F1
4. **Tests appariés** sur le même test set : McNemar (accuracy) et bootstrap apparié (macro-F1)

**Règles**
- Tous les choix (meilleure epoch, meilleur rang LoRA) se font sur la **validation**.
- Le test sert uniquement à rapporter les scores des modèles et à les comparer.
- Les résultats sont enregistrés dans Google Drive (`MyDrive/ledgar_A/`). Si Colab se déconnecte, relance toutes les cellules : les runs déjà terminés ne sont pas refaits.

**Avant de lancer** : Exécution → Modifier le type d'exécution → **GPU T4**.

## 1 · Installation : repo de l'équipe, GPU et Google Drive

On part du repo de l'équipe pour réutiliser **le même chargement de données** que `01_eda` (`ledgar.data.load_ledgar`).
Remplace `REPO_URL` par l'adresse du repo. S'il est privé, ajoute ton token GitHub dans les Secrets Colab (🔑) sous le nom `GITHUB_TOKEN`.

In [ ]:
import os, sys, torch
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4"
print("GPU :", torch.cuda.get_device_name(0))

!pip install -q -U peft

# Même chargement que load_ledgar() de 01_eda : le jeu officiel LexGLUE
from datasets import load_dataset
def load_ledgar():
    ds = load_dataset("coastalcph/lex_glue", "ledgar")
    return ds, ds["train"].features["label"].names
def team_set_seed(seed=42):
    pass

from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/ledgar_A"
os.makedirs(OUT, exist_ok=True)

GPU : Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 21.5 MB/s eta 0:00:00
Mounted at /content/drive


## 2 · Configuration

**Pourquoi Legal-BERT *uncased* et pas CamemBERT ?** L'EDA a montré des textes parfois en majuscules (« T HIS A MENDMENT ») : un modèle *uncased* y est insensible. LEDGAR est en **anglais**, alors que CamemBERT est un modèle **français**. Legal-BERT a la même taille que BERT-base (~110 M de paramètres), mais il a été pré-entraîné sur des textes juridiques anglais, dont des contrats de la base **EDGAR de la SEC**, d'où viennent justement les clauses de LEDGAR.

In [ ]:
MODEL_NAME      = "nlpaueb/legal-bert-base-uncased"   # repli possible : "bert-base-uncased"
MAX_LENGTH      = 256      # décision de l'EDA : 86 % des clauses entières, attention 4x moins chère qu'à 512
SEED            = 42
BATCH_SIZE      = 16
EVAL_BATCH_SIZE = 64
EPOCHS          = 3
LR_FULL         = 2e-5     # fine-tuning complet : petit learning rate (S06)
LR_LORA         = 5e-4     # LoRA : learning rate plus élevé, peu de paramètres à apprendre
LORA_RANKS      = [4, 8, 16]
WARMUP_RATIO    = 0.1      # transformers 5 : warmup_steps < 1 = proportion des steps (S07)
WEIGHT_DECAY    = 0.01

## 3 · Données et tokenisation

In [ ]:
import json, time, gc, shutil
import numpy as np, pandas as pd
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)
from sklearn.metrics import accuracy_score, f1_score

team_set_seed()
set_seed(SEED)
raw, LABELS = load_ledgar()          # même chargement que 01_eda
N_CLASSES = len(LABELS)
assert len(raw["train"]) == 60000, "FAST_RUN actif : on veut les splits complets"
print(raw)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=MAX_LENGTH)

ds = raw.map(tokenize, batched=True, remove_columns=["text"])
collator = DataCollatorWithPadding(tok)   # padding dynamique, batch par batch
Y_VAL, Y_TEST = np.array(raw["validation"]["label"]), np.array(raw["test"]["label"])

# Décision de l'EDA (01_eda) : ~2,8 % du test est déjà dans le train (doublons exacts).
# On rapporte aussi la macro-F1 du test SANS ces clauses, avec la même normalisation que l'EDA.
norm = lambda t: " ".join(t.lower().split())   # = re.sub(r"\s+", " ", ...).strip() de 01_eda
train_norm = set(map(norm, raw["train"]["text"]))
TEST_LEAK = np.array([norm(t) in train_norm for t in raw["test"]["text"]])
print(f"clauses du test déjà présentes dans le train : {TEST_LEAK.mean():.1%} ({TEST_LEAK.sum()})")

README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

ledgar/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.9MB            

ledgar/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ledgar/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.31MB            

ledgar/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/60000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

clauses du test déjà présentes dans le train : 2.8% (282)


## 4 · Fonctions : métriques et lancement d'une expérience

In [ ]:
def scores(y_true, y_pred):
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro"))}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = logits[0] if isinstance(logits, tuple) else logits
    return scores(labels, np.argmax(logits, -1))

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def logits_of(trainer, dataset):
    p = trainer.predict(dataset).predictions
    return p[0] if isinstance(p, tuple) else p

def save_preds(path, y_true, probs):
    pd.DataFrame({"idx": np.arange(len(y_true)), "y_true": y_true,
                  "y_pred": probs.argmax(1), "confidence": probs.max(1)}).to_csv(path, index=False)

def run_experiment(name, lora_r=None, lr=LR_FULL, epochs=EPOCHS, smoke=False):
    """Entraîne un modèle (complet si lora_r=None, LoRA sinon), garde la meilleure epoch
    selon la macro-F1 de VALIDATION, puis enregistre métriques et prédictions dans Drive."""
    run_dir = f"{OUT}/{name}"
    if not smoke and os.path.exists(f"{run_dir}/metrics.json"):
        print(f"[{name}] déjà terminé : je recharge les résultats.")
        return json.load(open(f"{run_dir}/metrics.json"))

    set_seed(SEED)
    train_ds, val_ds = ds["train"], ds["validation"]
    if smoke:   # test rapide du pipeline : 2 000 clauses, 1 epoch
        train_ds = train_ds.shuffle(seed=SEED).select(range(2000))
        val_ds, epochs = val_ds.select(range(1000)), 1

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_CLASSES,
        id2label=dict(enumerate(LABELS)), label2id={n: i for i, n in enumerate(LABELS)})
    if lora_r:
        from peft import LoraConfig, TaskType, get_peft_model
        # SEQ_CLS : la tête de classification reste entièrement entraînée
        model = get_peft_model(model, LoraConfig(
            task_type=TaskType.SEQ_CLS, r=lora_r, lora_alpha=2 * lora_r,
            lora_dropout=0.1, target_modules=["query", "value"]))   # W_Q et W_V
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"[{name}] paramètres entraînables : {trainable:,} / {total:,} ({100 * trainable / total:.2f} %)")

    args = TrainingArguments(
        output_dir=f"/content/ckpt/{name}",          # checkpoints en local, pas dans Drive
        learning_rate=lr, num_train_epochs=epochs,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE,
        warmup_steps=WARMUP_RATIO, weight_decay=WEIGHT_DECAY, lr_scheduler_type="linear",
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        fp16=True,   # précision mixte sur GPU
        logging_steps=200, report_to="none", seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      processing_class=tok, data_collator=collator, compute_metrics=compute_metrics)

    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    train_out = trainer.train()
    train_min = (time.time() - t0) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 1e9

    if smoke:
        est = train_out.metrics["train_runtime"] / 2000 * len(ds["train"]) / 60
        print(f"\nSMOKE TEST OK · validation : {trainer.evaluate()}")
        print(f"Estimation pour 1 epoch complète : ~{est:.0f} min "
              f"(soit ~{est * EPOCHS:.0f} min pour {EPOCHS} epochs) · pic mémoire GPU {peak_gb:.1f} Go")
        del trainer, model; gc.collect(); torch.cuda.empty_cache()
        return None

    os.makedirs(run_dir, exist_ok=True)
    # Prédictions avec le meilleur checkpoint (choisi sur la validation)
    val_probs = softmax(logits_of(trainer, ds["validation"]))
    t1 = time.time()
    test_probs = softmax(logits_of(trainer, ds["test"]))
    test_infer_s = time.time() - t1

    save_preds(f"{run_dir}/preds_val.csv", Y_VAL, val_probs)
    save_preds(f"{run_dir}/preds_test.csv", Y_TEST, test_probs)
    np.save(f"{run_dir}/probs_test.npy", test_probs.astype(np.float16))   # pour la calibration (volet C)
    json.dump(trainer.state.log_history, open(f"{run_dir}/log_history.json", "w"), indent=2)
    trainer.save_model(f"{run_dir}/model")   # modèle complet ou adaptateur LoRA

    metrics = {
        "run": name, "model": MODEL_NAME, "lora_r": lora_r, "lr": lr, "epochs": epochs,
        "max_length": MAX_LENGTH, "trainable_params": trainable, "total_params": total,
        "trainable_pct": round(100 * trainable / total, 3),
        "train_time_min": round(train_min, 1), "peak_gpu_mem_gb": round(peak_gb, 2),
        "test_inference_s": round(test_infer_s, 1),
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "validation": scores(Y_VAL, val_probs.argmax(1)),
        "test": scores(Y_TEST, test_probs.argmax(1)),
        "test_dedup": scores(Y_TEST[~TEST_LEAK], test_probs.argmax(1)[~TEST_LEAK]),
    }
    json.dump(metrics, open(f"{run_dir}/metrics.json", "w"), indent=2)
    print(json.dumps(metrics, indent=2))

    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(f"/content/ckpt/{name}", ignore_errors=True)
    return metrics

## 5 · Smoke test


In [ ]:
run_experiment("smoke", smoke=True)

[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

...

model.safetensors: downloading bytes:           |  0.00B            

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,4.229887,0.196000,0.027648


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
No log,4.229887,1,0.196000,0.027648



SMOKE TEST OK · validation : {'eval_loss': 4.229886531829834, 'eval_accuracy': 0.196, 'eval_macro_f1': 0.027647946250599874}
Estimation pour 1 epoch complète : ~22 min (soit ~67 min pour 3 epochs) · pic mémoire GPU 3.3 Go


## 6 · Fine-tuning complet

⚠️ **L'hypothèse doit être commitée dans le README avant de lancer cette cellule** (jalon m4).

In [ ]:
full = run_experiment("full_ft", lora_r=None, lr=LR_FULL)

[full_ft] déjà terminé : je recharge les résultats.


## 7 · LoRA : r = 4, 8, 16

In [ ]:
!pip uninstall -y -q torchao

In [11]:
lora = {r: run_experiment(f"lora_r{r}", lora_r=r, lr=LR_LORA) for r in LORA_RANKS}

[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

[lora_r4] paramètres entraînables : 224,356 / 109,783,496 (0.20 %)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.720213,0.648155,0.816400,0.707087
2,0.563081,0.570487,0.839900,0.742350


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

[lora_r16] paramètres entraînables : 666,724 / 110,225,864 (0.60 %)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.714510,0.643137,0.818100,0.710616
2,0.548913,0.553973,0.845900,0.750744
3,0.429341,0.516908,0.857000,0.767503


{
  "run": "lora_r16",
  "model": "nlpaueb/legal-bert-base-uncased",
  "lora_r": 16,
  "lr": 0.0005,
  "epochs": 3,
  "max_length": 256,
  "trainable_params": 666724,
  "total_params": 110225864,
  "trainable_pct": 0.605,
  "train_time_min": 33.9,
  "peak_gpu_mem_gb": 1.98,
  "test_inference_s": 44.2,
  "best_checkpoint": "/content/ckpt/lora_r16/checkpoint-11250",
  "validation": {
    "accuracy": 0.857,
    "macro_f1": 0.7675030341679472
  },
  "test": {
    "accuracy": 0.8578,
    "macro_f1": 0.7735671464966671
  },
  "test_dedup": {
    "accuracy": 0.8555258283597448,
    "macro_f1": 0.7715747088845964
  }
}


## 8 · Baseline TF-IDF (pour les tests appariés)


In [12]:
def run_baseline(C=10.0, class_weight="balanced"):
    run_dir = f"{OUT}/baseline"
    if os.path.exists(f"{run_dir}/preds_test.csv"):
        print("Baseline déjà disponible."); return
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    os.makedirs(run_dir, exist_ok=True)
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, sublinear_tf=True)
    Xtr = vec.fit_transform(raw["train"]["text"])       # fit sur le train uniquement
    t0 = time.time()
    clf = LogisticRegression(C=C, class_weight=class_weight, max_iter=2000).fit(Xtr, raw["train"]["label"])
    train_min = (time.time() - t0) / 60
    val_probs = clf.predict_proba(vec.transform(raw["validation"]["text"]))
    test_probs = clf.predict_proba(vec.transform(raw["test"]["text"]))
    save_preds(f"{run_dir}/preds_val.csv", Y_VAL, val_probs)
    save_preds(f"{run_dir}/preds_test.csv", Y_TEST, test_probs)
    json.dump({"run": "baseline", "model": f"TF-IDF + LogReg (C={C}, {class_weight})",
               "lora_r": None, "trainable_params": int(clf.coef_.size + clf.intercept_.size),
               "total_params": int(clf.coef_.size + clf.intercept_.size), "trainable_pct": 100.0,
               "train_time_min": round(train_min, 1), "peak_gpu_mem_gb": 0.0,
               "validation": scores(Y_VAL, val_probs.argmax(1)),
               "test": scores(Y_TEST, test_probs.argmax(1)),
               "test_dedup": scores(Y_TEST[~TEST_LEAK], test_probs.argmax(1)[~TEST_LEAK])},
              open(f"{run_dir}/metrics.json", "w"), indent=2)
    print(json.load(open(f"{run_dir}/metrics.json")))

run_baseline()

{'run': 'baseline', 'model': 'TF-IDF + LogReg (C=10.0, balanced)', 'lora_r': None, 'trainable_params': 17347800, 'total_params': 17347800, 'trainable_pct': 100.0, 'train_time_min': 8.3, 'peak_gpu_mem_gb': 0.0, 'validation': {'accuracy': 0.8711, 'macro_f1': 0.8138070646261177}, 'test': {'accuracy': 0.8685, 'macro_f1': 0.8233585699198852}, 'test_dedup': {'accuracy': 0.8649927968717843, 'macro_f1': 0.8197037991652028}}


## 9 · Tableau de compromis

In [13]:
RUNS = ["baseline", "full_ft"] + [f"lora_r{r}" for r in LORA_RANKS]
rows = []
for name in RUNS:
    p = f"{OUT}/{name}/metrics.json"
    if not os.path.exists(p):
        continue
    m = json.load(open(p))
    rows.append({
        "Modèle": name,
        "Params entraînables": f"{m['trainable_params']:,}",
        "% entraînés": m["trainable_pct"],
        "Temps (min)": m["train_time_min"],
        "Mémoire GPU (Go)": m["peak_gpu_mem_gb"],
        "Val macro-F1": round(100 * m["validation"]["macro_f1"], 2),
        "Test accuracy": round(100 * m["test"]["accuracy"], 2),
        "Test macro-F1": round(100 * m["test"]["macro_f1"], 2),
        "Test macro-F1 sans doublons": round(100 * m["test_dedup"]["macro_f1"], 2),
    })
table = pd.DataFrame(rows)
table.to_csv(f"{OUT}/tradeoff_table.csv", index=False)
display(table)
try:
    print(table.to_markdown(index=False))   # à coller dans le README
except ImportError:
    pass

,Modèle,Params entraînables,% entraînés,Temps (min),Mémoire GPU (Go),Val macro-F1,Test accuracy,Test macro-F1,Test macro-F1 sans doublons
0,baseline,"17,347,800",100.000,8.3,0.00,81.38,86.85,82.34,81.97
1,full_ft,"109,559,140",100.000,42.4,3.26,79.24,86.86,77.94,77.68
2,lora_r4,"224,356",0.204,34.9,1.97,74.94,84.86,75.98,75.89
3,lora_r8,"371,812",0.338,33.9,1.98,76.84,85.28,76.84,76.74
4,lora_r16,"666,724",0.605,33.9,1.98,76.75,85.78,77.36,77.16


| Modèle   | Params entraînables   |   % entraînés |   Temps (min) |   Mémoire GPU (Go) |   Val macro-F1 |   Test accuracy |   Test macro-F1 |   Test macro-F1 sans doublons |
|:---------|:----------------------|--------------:|--------------:|-------------------:|---------------:|----------------:|----------------:|------------------------------:|
| baseline | 17,347,800            |       100     |           8.3 |               0    |          81.38 |           86.85 |           82.34 |                         81.97 |
| full_ft  | 109,559,140           |       100     |          42.4 |               3.26 |          79.24 |           86.86 |           77.94 |                         77.68 |
| lora_r4  | 224,356               |         0.204 |          34.9 |               1.97 |          74.94 |           84.86 |           75.98 |                         75.89 |
| lora_r8  | 371,812               |         0.338 |          33.9 |               1.98 |          76.84 |           85.28 | 

## 10 · Tests appariés (même test set)

- **McNemar** : compare les accuracies en ne regardant que les clauses où un seul des deux modèles a raison.
- **Bootstrap apparié** : on rééchantillonne 1 000 fois le test set (le même tirage pour les deux modèles) et on regarde si l'intervalle à 95 % de la différence de macro-F1 contient 0.

Le meilleur LoRA est choisi sur la **validation**, pas sur le test.

In [14]:
from scipy.stats import binomtest, chi2

def mcnemar(y, pa, pb):
    ok_a, ok_b = pa == y, pb == y
    b, c = int(np.sum(ok_a & ~ok_b)), int(np.sum(~ok_a & ok_b))
    if b + c < 25:
        p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
        return {"b_only_A_right": b, "c_only_B_right": c, "test": "exact binomial", "p_value": float(p)}
    stat = (abs(b - c) - 1) ** 2 / (b + c)
    return {"b_only_A_right": b, "c_only_B_right": c, "test": "chi2 (continuity corr.)",
            "chi2": float(stat), "p_value": float(chi2.sf(stat, df=1))}

def paired_bootstrap_f1(y, pa, pb, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    n, diffs = len(y), np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)          # même tirage pour A et B
        diffs[i] = (f1_score(y[idx], pa[idx], average="macro")
                    - f1_score(y[idx], pb[idx], average="macro"))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"delta_macro_f1_pts": round(100 * (f1_score(y, pa, average="macro")
                                               - f1_score(y, pb, average="macro")), 2),
            "ci95_pts": [round(100 * lo, 2), round(100 * hi, 2)],
            "significant": bool(lo > 0 or hi < 0)}

def test_preds(name):
    return pd.read_csv(f"{OUT}/{name}/preds_test.csv")["y_pred"].values

done = [n for n in RUNS if os.path.exists(f"{OUT}/{n}/preds_test.csv")]
loras = [n for n in done if n.startswith("lora")]
best_lora = max(loras, key=lambda n: json.load(open(f"{OUT}/{n}/metrics.json"))["validation"]["macro_f1"]) if loras else None
print("Meilleur LoRA (choisi sur la validation) :", best_lora)

pairs = [("full_ft", "baseline"), ("full_ft", best_lora), (best_lora, "baseline")]
stats = {}
for a, b in pairs:
    if a in done and b in done:
        pa, pb = test_preds(a), test_preds(b)
        stats[f"{a} vs {b}"] = {"mcnemar": mcnemar(Y_TEST, pa, pb),
                                "bootstrap": paired_bootstrap_f1(Y_TEST, pa, pb)}
        print(f"\n{a} vs {b}\n", json.dumps(stats[f'{a} vs {b}'], indent=2))
json.dump(stats, open(f"{OUT}/paired_tests.json", "w"), indent=2)

Meilleur LoRA (choisi sur la validation) : lora_r8

full_ft vs baseline
 {
  "mcnemar": {
    "b_only_A_right": 456,
    "c_only_B_right": 455,
    "test": "chi2 (continuity corr.)",
    "chi2": 0.0,
    "p_value": 1.0
  },
  "bootstrap": {
    "delta_macro_f1_pts": -4.4,
    "ci95_pts": [
      -5.44,
      -2.78
    ],
    "significant": true
  }
}

full_ft vs lora_r8
 {
  "mcnemar": {
    "b_only_A_right": 338,
    "c_only_B_right": 180,
    "test": "chi2 (continuity corr.)",
    "chi2": 47.584942084942085,
    "p_value": 5.267150910890984e-12
  },
  "bootstrap": {
    "delta_macro_f1_pts": 1.1,
    "ci95_pts": [
      0.23,
      2.52
    ],
    "significant": true
  }
}

lora_r8 vs baseline
 {
  "mcnemar": {
    "b_only_A_right": 446,
    "c_only_B_right": 603,
    "test": "chi2 (continuity corr.)",
    "chi2": 23.199237368922784,
    "p_value": 1.4605523840311671e-06
  },
  "bootstrap": {
    "delta_macro_f1_pts": -5.49,
    "ci95_pts": [
      -6.81,
      -4.05
    ],
    "sign

## 11 · Export vers le repo de l'équipe


In [15]:
from pathlib import Path
TAB, PRED = Path("results/tables"), Path("results/predictions")
TAB.mkdir(parents=True, exist_ok=True); PRED.mkdir(parents=True, exist_ok=True)
shutil.copy(f"{OUT}/tradeoff_table.csv", TAB / "A_tradeoff_table.csv")
shutil.copy(f"{OUT}/paired_tests.json", TAB / "A_paired_tests.json")
for name in RUNS:
    d = f"{OUT}/{name}"
    if os.path.exists(f"{d}/metrics.json"):
        shutil.copy(f"{d}/metrics.json", TAB / f"A_metrics_{name}.json")
        shutil.copy(f"{d}/preds_test.csv", PRED / f"{name}_test.csv")
        shutil.copy(f"{d}/preds_val.csv", PRED / f"{name}_val.csv")
!git status --short

fatal: not a git repository (or any of the parent directories): .git


In [16]:
# Où BERT perd-il ? F1 par groupe de fréquence (mêmes tiers que 01_eda)
counts = np.bincount(raw["train"]["label"], minlength=N_CLASSES)
rank = np.argsort(np.argsort(-counts))
group = np.where(rank < 34, "1-fréquente", np.where(rank < 67, "2-moyenne", "3-rare"))
models = ["baseline", "full_ft", "lora_r8"]
f1s = {m: f1_score(Y_TEST, pd.read_csv(f"{OUT}/{m}/preds_test.csv").y_pred.values,
                   average=None, labels=range(N_CLASSES), zero_division=0) for m in models}
by_group = pd.DataFrame({m: pd.Series(f1s[m]).groupby(group).mean() * 100 for m in models}).round(1)
display(by_group)
by_group.to_csv(f"{OUT}/f1_by_freq_group.csv")

diff = pd.DataFrame({"classe": LABELS, "n_train": counts,
                     "F1 baseline": f1s["baseline"] * 100, "F1 BERT": f1s["full_ft"] * 100})
diff["écart"] = diff["F1 BERT"] - diff["F1 baseline"]
display(diff.sort_values("écart").head(10).round(1))
diff.to_csv(f"{OUT}/f1_per_class_baseline_vs_bert.csv", index=False)

,baseline,full_ft,lora_r8
1-fréquente,88.9,89.5,87.9
2-moyenne,83.4,80.7,78.5
3-rare,74.5,63.2,63.8


,classe,n_train,F1 baseline,F1 BERT,écart
14,Books,23,100.0,0.0,-100.0
72,Qualifications,47,75.0,0.0,-75.0
4,Applicable Laws,352,32.4,0.0,-32.4
8,Assigns,31,28.6,0.0,-28.6
52,Integration,304,78.6,54.8,-23.8
28,Defined Terms,354,71.0,48.0,-23.0
25,Costs,168,66.7,45.7,-21.0
5,Approvals,178,63.8,45.3,-18.5
50,Indemnity,271,67.9,51.1,-16.9
37,Enforcements,201,64.9,48.4,-16.5


In [17]:
from google.colab import userdata
import os, shutil, glob

REPO   = "saouss-07/ledgar-contract-review"
BRANCH = "lchapoton-patch-1"
token  = userdata.get("GITHUB_TOKEN")

# 1. Récupérer la branche
!rm -rf /content/push
!git clone -q -b {BRANCH} https://{token}@github.com/{REPO}.git /content/push

# 2. Copier le notebook (version enregistrée dans le Drive)
nb = glob.glob("/content/drive/MyDrive/**/A_finetune_bert_lora*.ipynb", recursive=True)
print("notebook trouvé :", nb)
os.makedirs("/content/push/notebooks", exist_ok=True)
shutil.copy(nb[0], "/content/push/notebooks/A_finetune_bert_lora.ipynb")

# 3. Copier les résultats
T, P = "/content/push/results/tables", "/content/push/results/predictions"
os.makedirs(T, exist_ok=True); os.makedirs(P, exist_ok=True)
for f in ["tradeoff_table.csv", "paired_tests.json", "f1_by_freq_group.csv", "f1_per_class_baseline_vs_bert.csv"]:
    shutil.copy(f"{OUT}/{f}", f"{T}/A_{f}")
for run in ["baseline", "full_ft", "lora_r4", "lora_r8", "lora_r16"]:
    shutil.copy(f"{OUT}/{run}/metrics.json", f"{T}/A_metrics_{run}.json")
    shutil.copy(f"{OUT}/{run}/preds_test.csv", f"{P}/A_{run}_test.csv")
    shutil.copy(f"{OUT}/{run}/preds_val.csv", f"{P}/A_{run}_val.csv")

# 4. Commit et push sur la branche
%cd /content/push
!git config user.name "Léa"
!git config user.email "leachapoton75@gmail.com"
!git add -f notebooks/A_finetune_bert_lora.ipynb results/tables results/predictions
!git status --short
!git commit -q -m "Volet A : notebook fine-tuning + LoRA, tableau de compromis, tests appariés, prédictions"
!git push -q origin {BRANCH}
!git log --oneline -3
%cd /content

notebook trouvé : ['/content/drive/MyDrive/Colab Notebooks/A_finetune_bert_lora (1).ipynb', '/content/drive/MyDrive/Colab Notebooks/A_finetune_bert_lora.ipynb']
/content/push
A  notebooks/A_finetune_bert_lora.ipynb
A  results/predictions/A_baseline_test.csv
A  results/predictions/A_baseline_val.csv
A  results/predictions/A_full_ft_test.csv
A  results/predictions/A_full_ft_val.csv
A  results/predictions/A_lora_r16_test.csv
A  results/predictions/A_lora_r16_val.csv
A  results/predictions/A_lora_r4_test.csv
A  results/predictions/A_lora_r4_val.csv
A  results/predictions/A_lora_r8_test.csv
A  results/predictions/A_lora_r8_val.csv
A  results/tables/A_f1_by_freq_group.csv
A  results/tables/A_f1_per_class_baseline_vs_bert.csv
A  results/tables/A_metrics_baseline.json
A  results/tables/A_metrics_full_ft.json
A  results/tables/A_metrics_lora_r16.json
A  results/tables/A_metrics_lora_r4.json
A  results/tables/A_metrics_lora_r8.json
A  results/tables/A_paired_tests.json
A  results/tables/A_tradeo